In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

from utils_firerisk import (
    REGION, add_lag_features, build_panel, calendar_features, cluster_events, daily_from_hourly,
    load_oni, nesterov_series, summarize_events,
)

DATA = Path("../data")
RAW, INTERIM, PROCESSED = DATA / "raw", DATA / "interim", DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

START_DATE, END_DATE = "2012-01-20", "2025-12-31"
PRODUCT = "VIIRS_SNPP_SP"
TREE_FRAC_MIN = 0.5
BIG_PIXELS, BIG_DAYS = 5, 2
A0_GRID = [5, 6, 7, 8, 9, 10]
CLIM_END = "2023-12-31"
RUN_SUBSET = None

In [2]:
PROV = pd.read_csv(INTERIM / "prov_table.csv")
static = pd.read_csv(INTERIM / "static_prov.csv")
if RUN_SUBSET:
    PROV = PROV[PROV["key"].isin(RUN_SUBSET)].reset_index(drop=True)

HOTSPOTS_FILE = INTERIM / "hotspots_enriched.csv"
pts = pd.read_csv(HOTSPOTS_FILE, parse_dates=["ltime", "ldate"])
pts = pts[pts["prov"].isin(PROV["key"])].copy()
pts["forest"] = pts["tree_frac"] >= TREE_FRAC_MIN
pts["conf_ok"] = pts["confidence"].astype(str).str.lower().isin(["n", "h", "nominal", "high"])
print(len(PROV), "tỉnh |", len(pts), "điểm cháy | tỉnh có đặc trưng tĩnh:", int(PROV["key"].isin(static["key"]).sum()))

63 tỉnh | 947287 điểm cháy | tỉnh có đặc trưng tĩnh: 63


In [3]:
events, forest_hotspots = {}, {}
for key in PROV["key"]:
    selected = pts[(pts["prov"] == key) & pts["forest"] & pts["conf_ok"]]
    clustered = cluster_events(selected[["latitude", "longitude", "ldate", "frp"]])
    forest_hotspots[key] = clustered
    events[key] = summarize_events(clustered)
print(pd.Series({k: len(v) for k, v in events.items()}).describe().round(1).to_dict())

{'count': 63.0, 'mean': 3689.3, 'std': 4440.6, 'min': 1.0, '25%': 270.5, '50%': 2173.0, '75%': 5564.0, 'max': 16124.0}


Số sự kiện cháy rừng mỗi tỉnh: trung bình 3.689, trung vị 2.173, thấp nhất 1 và cao nhất 16.124. Chênh lệch rất lớn giữa tỉnh có rừng và tỉnh đồng bằng.

In [4]:
WX_COLUMNS = ["time", "temperature_2m", "dew_point_2m", "precipitation"]


def load_weather(key):
    years = range(pd.Timestamp(START_DATE).year - 1, pd.Timestamp(END_DATE).year + 1)
    frames = [pd.read_csv(RAW / "wx" / f"{key}_{year}.csv", usecols=WX_COLUMNS) for year in years]
    return daily_from_hourly(pd.concat(frames, ignore_index=True))


weather = {key: load_weather(key) for key in PROV["key"]}
days_per_province = len(next(iter(weather.values())))
missing_t13 = int(sum(w["T13"].isna().sum() for w in weather.values()))
print("Thời tiết ngày:", len(weather), "tỉnh | ngày mỗi tỉnh:", days_per_province, "| thiếu T13:", missing_t13)

Thời tiết ngày: 63 tỉnh | ngày mỗi tỉnh: 5479 | thiếu T13: 0


In [5]:
def add_nesterov(d):
    d = d.copy()
    for a0 in A0_GRID:
        for mode in ["linear", "binary"]:
            for name, column in [("Dsat", "D13_hpa"), ("Ddew", "dd13_c")]:
                d[f"P_{mode}{a0}_{name}"] = nesterov_series(d["T13"], d[column], d["rain_cal"], a0, mode)
    d["P_alt19_linear7_Dsat"] = nesterov_series(d["T13"], d["D13_hpa"], d["rain_19_19"], 7, "linear")
    d["P_alt19_binary5_Dsat"] = nesterov_series(d["T13"], d["D13_hpa"], d["rain_19_19"], 5, "binary")
    d["P_nesterov_3mm"] = nesterov_series(d["T13"], d["dd13_c"], d["rain_cal"], 3.05, "binary")
    return d


for key in weather:
    weather[key] = add_lag_features(add_nesterov(weather[key]), CLIM_END)

oni = load_oni(INTERIM / "oni.csv")
print("ONI:", oni.index.min().date(), "->", oni.index.max().date())


def oni_columns(index):
    months = pd.DatetimeIndex(index).to_period("M").to_timestamp()
    return oni.reindex(months).values, oni.shift(3).reindex(months).values

ONI: 1950-01-01 -> 2026-09-01


Bảng giữ 27 biến thể của chỉ số Nesterov: 26 biến thể do nhóm xây dựng (ngưỡng mưa đặt lại 5 đến 10 mm) và một biến thể gần với chỉ số Nesterov gốc (P_nesterov_3mm: đặt lại khi mưa ngày lớn hơn 3 mm, cộng dồn nhiệt độ nhân độ thiếu hụt điểm sương lúc 13 giờ).

In [6]:
STATIC_COLUMNS = ["area_km2", "forest_share", "crop_share", "grass_share", "built_share", "rep_elev_m", "rep_lat", "rep_lon"]
win0, win1 = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)

parts, info = [], []
for _, prov in PROV.iterrows():
    key, region = prov["key"], prov["region"]

    hot_all = pts[(pts["prov"] == key) & pts["conf_ok"]]
    panel = build_panel(weather[key].loc[win0:win1], hot_all, forest_hotspots[key], events[key], BIG_PIXELS, BIG_DAYS)
    panel = pd.concat([panel, calendar_features(panel.index)], axis=1)
    panel["oni"], panel["oni_lag3"] = oni_columns(panel.index)

    province_static = static[static["key"] == key].iloc[0]
    for column in STATIC_COLUMNS:
        panel[column] = province_static[column]
    panel.insert(0, "prov", key)
    panel.insert(1, "region", region)

    info.append({"prov": key, "region": region, "rows": len(panel)})
    parts.append(panel)

panel = pd.concat(parts).reset_index()
panel.to_csv(PROCESSED / "panel.csv", index=False)

print(pd.DataFrame(info).head(10).to_string(index=False))
print("panel:", panel.shape)

           prov region  rows
       an giang  DBSCL  5095
ba ria vung tau    DNB  5095
      bac giang     DB  5095
        bac kan     DB  5095
       bac lieu  DBSCL  5095
       bac ninh   DBSH  5095
        ben tre  DBSCL  5095
      binh dinh    NTB  5095
     binh duong    DNB  5095
     binh phuoc    DNB  5095
panel: (320985, 74)


Bảng gộp có 63 tỉnh x 5.095 ngày = 320.985 dòng, 75 cột.

In [7]:
q = pts.copy()
q["year"] = q["ltime"].dt.year
q["region"] = q["prov"].map(REGION)

print("Giờ địa phương trung vị của lần bay qua, theo năm")
print(q.groupby(["year", "daynight"])["lhour"].median().unstack().round(1).T.to_string())

print("\nTỉ lệ độ tin cậy theo năm")
print(q.groupby("year")["confidence"].value_counts(normalize=True).unstack().round(3).to_string())

print("\nLớp phủ tại điểm cháy theo vùng (10 rừng, 40 nông nghiệp, 30 cỏ, 20 cây bụi)")
print(q.groupby("region")["lc_center"].value_counts(normalize=True).unstack().round(3).fillna(0).to_string())

print("\nĐiểm cháy theo tháng và vùng")
print(q[q["conf_ok"]].groupby(["region", q["ltime"].dt.month]).size().unstack(fill_value=0).to_string())

print("\nTỉ lệ thiếu thời tiết theo vùng")
print(panel.groupby("region")[["T13", "D13_hpa", "rain_cal"]].apply(lambda g: g.isna().mean()).round(4).to_string())

gate2 = q[q["forest"] & q["conf_ok"]].groupby(["prov", "year"]).agg(
    hotspot_days=("ldate", "nunique"), hotspots=("ldate", "size")
).reset_index()
gate2.to_csv(PROCESSED / "gate2_province_year.csv", index=False)

Giờ địa phương trung vị của lần bay qua, theo năm
year      2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
daynight                                                                                          
D         13.0  13.0  13.0  13.0  13.0  13.0  13.0  13.0  13.0  13.0  13.0  13.0  13.0  13.0   NaN
N          1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0

Tỉ lệ độ tin cậy theo năm
confidence      h      l      n
year                           
2012        0.031  0.074  0.895
2013        0.037  0.075  0.889
2014        0.038  0.094  0.868
2015        0.041  0.098  0.861
2016        0.044  0.083  0.874
2017        0.026  0.082  0.892
2018        0.031  0.079  0.891
2019        0.034  0.086  0.880
2020        0.034  0.085  0.881
2021        0.031  0.079  0.890
2022        0.030  0.050  0.920
2023        0.034  0.075  0.891
2024        0.035  0.084  0.882
2025        0.028  0.057  0.915
2026          N

lc_center   0      10     20     30     40     50     60     80     90     95
region                                                                       
BTB        0.0  0.783  0.000  0.089  0.040  0.070  0.011  0.007  0.001  0.000
DB         0.0  0.825  0.000  0.115  0.025  0.026  0.004  0.005  0.000  0.001
DBSCL      0.0  0.074  0.001  0.016  0.867  0.028  0.003  0.006  0.004  0.000
DBSH       0.0  0.193  0.000  0.040  0.102  0.463  0.169  0.031  0.000  0.000
DNB        0.0  0.466  0.000  0.130  0.215  0.136  0.031  0.021  0.001  0.001
NTB        0.0  0.740  0.000  0.160  0.069  0.019  0.007  0.004  0.000  0.000
TB         0.0  0.549  0.000  0.408  0.039  0.001  0.001  0.001  0.000  0.000
TN         0.0  0.501  0.000  0.386  0.105  0.004  0.001  0.003  0.000  0.000

Điểm cháy theo tháng và vùng


ltime      1      2      3      4      5     6     7      8     9     10    11    12
region                                                                              
BTB      5314   6102   9983  14564  13145  9178  3594   4033  4516  4027  4677  4146
DB       7448   8192   9716   9183   6502  2050   852    747  1258  2432  3639  7631
DBSCL    7381  15182  48290  23374   5713  2774  2532   3983  1664   338  2219  4455
DBSH     1272    597    692    864    965  1073  1024   1068  1449  1845  1743  1770
DNB      4047   4869   6875   5226   2679  1572  1037   1307   760   901  1117  2101
NTB      6585  13769  20281  20775  16228  9774  6707  12189  9627  5380  2600  1875
TB       7729  29278  69821  49442   8521   448   113    156   523  2542  5338  5324
TN      29332  60894  94094  48407  10580  3196  1427   1753   884   907   900  5447

Tỉ lệ thiếu thời tiết theo vùng
        T13  D13_hpa  rain_cal
region                        
BTB     0.0      0.0       0.0
DB      0.0      0.0    

Giờ vệ tinh bay qua ổn định suốt 14 năm (ban ngày 13h, ban đêm 1h), không có dấu hiệu trôi quỹ đạo. Tỉ lệ độ tin cậy cũng ổn định (n khoảng 86 đến 92%, h 3 đến 4%, l 5 đến 10%), riêng l thấp bất thường ở 2022 và 2025. Rừng chiếm 78 đến 83% điểm cháy ở Bắc Trung Bộ và Đông Bắc, 74% ở Nam Trung Bộ, 50 đến 55% ở Tây Bắc và Tây Nguyên. Đồng bằng sông Cửu Long 87% là đất nông nghiệp, đồng bằng sông Hồng 46% là đô thị. Thời tiết không thiếu giá trị nào ở T13, D13 và mưa trong cả 8 vùng.

In [8]:
def sha256_prefix(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()[:16]


tracked = {
    "hotspots_enriched.csv": HOTSPOTS_FILE,
    "gate2_province_year.csv": PROCESSED / "gate2_province_year.csv",
    "static_prov.csv": INTERIM / "static_prov.csv",
}
settings = {
    "created": pd.Timestamp.now("UTC").isoformat(),
    "product": PRODUCT,
    "start": START_DATE,
    "end": END_DATE,
    "tree_frac_min": TREE_FRAC_MIN,
    "big_pixels": BIG_PIXELS,
    "big_days": BIG_DAYS,
    "a0_grid": json.dumps(A0_GRID),
    "n_provinces": len(PROV),
    "run_subset": json.dumps(RUN_SUBSET),
    "n_hotspots_in_provinces": len(pts),
    "rows_panel": len(panel),
}
manifest = pd.DataFrame(
    [("setting", k, v) for k, v in settings.items()]
    + [("sha256", name, sha256_prefix(path)) for name, path in tracked.items() if path.exists()],
    columns=["type", "name", "value"],
)
manifest.to_csv(PROCESSED / "manifest.csv", index=False)
manifest

,type,name,value
0,setting,created,2026-11-09T08:03:55.324254+00:00
1,setting,product,VIIRS_SNPP_SP
2,setting,start,2012-01-20
3,setting,end,2025-12-31
4,setting,tree_frac_min,0.5
5,setting,big_pixels,5
6,setting,big_days,2
7,setting,a0_grid,"[5, 6, 7, 8, 9, 10]"
8,setting,n_provinces,63
9,setting,run_subset,null
